
### TRÁFICO GA4
###### El siguiente código consulta datos de tráfico utilizando una api de Google Analytics 4. Se carga de forma diaria pero su partición es mensual. Tiempo de ejecución estimado de 6 minutos.

In [0]:
import re, os, subprocess, time, itertools, json
import pandas as pd
import numpy as np
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from joblib import dump, load
from datetime import date, datetime, timedelta
from delta.tables import DeltaTable
from tenacity import retry, wait_incrementing, stop_after_attempt
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.oauth2 import service_account
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    Filter,
    FilterExpression,
    FilterExpressionList,
    RunReportRequest,
)

In [0]:
#adls_container = dbutils.widgets.text("adls_container", "abfss://ingestas@stbigdatadev02.dfs.core.windows.net")
#catalog_control = dbutils.widgets.text("catalog_control", "bidesarrollo.control.control_ingestas")
#dir_adls = dbutils.widgets.text("dir_adls", "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/trafico/detalle_navegacion/google/web_sessions/")
#endtime_nifi = dbutils.widgets.text("endtime_nifi", "20250307 120857")
#key_secret = dbutils.widgets.text("key_secret", "sec-json-glowing-ocean-338019")
#pipelineRunId = dbutils.widgets.text("pipelineRunId", "89cfd193-83d4-4cf9-a29f-f78ce83ff9c1")
#property_id = dbutils.widgets.text("property_id", "313644144")
#catalogo = dbutils.widgets.text("catalogo", "bi_ingestas")
#schema = dbutils.widgets.text("schema", "raw_trafico")
#table_name = dbutils.widgets.text("table_name", "web_sessions_movistar_b2c")
#scope = dbutils.widgets.text("scope", "secrets-ingestas")
#starttime_nifi = dbutils.widgets.text("starttime_nifi", "20250307 120857")
#timestampFormat = dbutils.widgets.text("timestampFormat", "yyyyMMddHHmmssSSS")
#original_file_date = dbutils.widgets.text("original_file_date", "2024-05-17T13:03:41Z")




In [0]:
dir_adls        = dbutils.widgets.get("dir_adls") 
catalogo        = dbutils.widgets.get("catalogo")
schema          = dbutils.widgets.get("schema")
table_name      = dbutils.widgets.get("table_name")
scope           = dbutils.widgets.get("scope")
key_secret      = dbutils.widgets.get("key_secret")
property_id     = dbutils.widgets.get("property_id")

tabla_vw        = f"{catalogo}.{schema}.{table_name}"
#print (tabla_vw)
process_name    = f"spark_{table_name}"

timestampFormat = (dbutils.widgets.get("timestampFormat") or "yyyy-MM-dd'T'HH:mm:ss[.SSS][XXX]").replace("|",":")
starttime_nifi = dbutils.widgets.get("starttime_nifi")
#print ("Obtenido starttime_nifi: ", starttime_nifi)
starttime_nifi_datetime = datetime.strptime(starttime_nifi, "%Y%m%d %H%M%S")
fecha_inicio_ejecucion = starttime_nifi_datetime.strftime("%Y-%m-%d")

endtime_nifi = dbutils.widgets.get("endtime_nifi")
endtime_nifi_datetime = datetime.strptime(endtime_nifi, "%Y%m%d %H%M%S")
pipelineRunId = dbutils.widgets.get("pipelineRunId")
catalog_control = dbutils.widgets.get("catalog_control")

#print ("fecha_inicio_ejecucion: ", fecha_inicio_ejecucion)

In [0]:
# 1. Definir Funciones
# Función con número de intento para tenacity
def log_attempt_number(retry_state): 
    """return the result of the last call attempt"""
    #print(f"Error! Retrying: {retry_state.attempt_number}...")

# Función para transformar datos json (respuesta de la API) a dataframe
def crear_dataframe(api_response):
    dimension_headers = [header.name for header in api_response.dimension_headers]
    metric_headers = [header.name for header in api_response.metric_headers]
    dimensions = []
    metrics = []
    for i in range(len(dimension_headers)):
        dimensions.append([row.dimension_values[i].value for row in api_response.rows])
    dimensions
    for i in range(len(metric_headers)):
        metrics.append([row.metric_values[i].value for row in api_response.rows])
    headers = dimension_headers, metric_headers
    headers = list(itertools.chain.from_iterable(headers))
    data = dimensions, metrics
    data = list(itertools.chain.from_iterable(data))
    df = pd.DataFrame(data)
    df = df.transpose()
    df.columns = headers
    return df

# Función para extraer tráfico y métrica interes de la API GA4 (respuesta en formato JSON)
@retry(wait=wait_incrementing(start=5, increment=5), stop=stop_after_attempt(10), after=log_attempt_number) # opcional
def query_ga(property_id, fecha_inicio, fecha_termino, dims, metrics, filtro):
    request_api = RunReportRequest(
        property=f"properties/{property_id}",
        date_ranges=[DateRange(start_date=fecha_inicio, end_date=fecha_termino)],
        limit=2500000,
        dimensions=[Dimension(name=dim) for dim in dims],
        metrics=[Metric(name=mets) for mets in metrics],
        dimension_filter=FilterExpression(
           or_group=FilterExpressionList(
              expressions=[
                FilterExpression(
                    filter=Filter(
                        field_name="pageLocation",
                        string_filter=Filter.StringFilter(value=filtro,
                                                            match_type= Filter.StringFilter.MatchType.PARTIAL_REGEXP),
                )),
                FilterExpression(
                    filter=Filter(
                        field_name="customEvent:virtualurl",
                        string_filter=Filter.StringFilter(value=filtro,
                                                            match_type= Filter.StringFilter.MatchType.PARTIAL_REGEXP),
                ))
        ])))
    data = client.run_report(request_api)
    return data

# Función para extraer CONVERSIONES de la API GA4 (respuesta en formato JSON)
@retry(wait=wait_incrementing(start=5, increment=5), stop=stop_after_attempt(10), after=log_attempt_number) # opcional
def query_ga_conversions(property_id, fecha_inicio, fecha_termino, dims, metrics):
    request_api = RunReportRequest(
        property=f"properties/{property_id}",
        date_ranges=[DateRange(start_date=fecha_inicio, end_date=fecha_termino)],
        limit=2500000,
        dimensions=[Dimension(name=dim) for dim in dims],
        metrics=[Metric(name=mets) for mets in metrics])
    data = client.run_report(request_api)
    return data



In [0]:
# 2. Credenciales y Autentificación
status = "[OK]"
desc_status = "[OK]"

try:
    secret_json = dbutils.secrets.get(scope = scope, key = key_secret)
    service_account_info = json.loads(secret_json)
    credentials = service_account.Credentials.from_service_account_info(service_account_info)
    client = BetaAnalyticsDataClient(credentials=credentials)
except Exception as e:
    status = "[ERROR]"
    desc_status = f"Error de credenciales: {e}" 
print (status)

In [0]:
# 4. Definición parametros Query
# Fechas para ejcutar 3 días hacia atrás 
fecha_fin_ejecucion = fecha_inicio_ejecucion
fecha_tmp_ejecucion = datetime.strptime(fecha_inicio_ejecucion, "%Y-%m-%d").date()

original_file_size = 0 
final_row_count = 0

# El código procesará los tres días anteriores a la fecha de ejecución.
for k in range(3): 
  fecha_tmp_ejecucion = fecha_tmp_ejecucion - timedelta(days=1)
  fecha_tmp_ejecucion_str = fecha_tmp_ejecucion.strftime("%Y-%m-%d")

  ## ID de Cuenta GA4 ##
  # property_id = "313644144"

  ## Dimensiones y métricas ##
  dims = ['date','sessionSource', 'sessionMedium', 'sessionCampaignName', 'sessionCustomChannelGroup:5602220125', 'deviceCategory']
  metrics = ['sessions', 'engagedSessions', 'totalUsers', 'activeUsers']
  metrics_sessions_conversion = ['keyEvents:Obj6_Leads_Hogar', 'keyEvents:Obj7_Leads_Movil', 'keyEvents:Obj3_Comprobante_Fullprice', 'keyEvents:Obj19_Leads_Recambio']
  metrics_sessions_interes = ['sessions']

  ## REGEXs para definir cada negocio ##
  filtro_hogar = "(?i).*(ww2\.movistar\.cl\/hogar\/((\?.*)?$|internet-fibra-optica|internet-hogar|test-velocidad|factibilidad-internet|pack-digital-netflix|pack-digital-netflix-tv-online|pack-duos-internet-television|pack-duos-internet-telefonia|pack-trios|telefonia|television|arma-tu-pack|arma-tu-plan|gaming).*|.*catalogo\.movistar\.cl\/servicioshogar\/(solicitudes|contratacion)\/datos\/index\/id\/.*|.*catalogo.movistar.cl\/vURL\/Hogar\/.*|.*ww2\.movistar\.cl\/(ofertas|ofertas.*)\/(internet-hogar|packs-hogar).*|.*ww2\.movistar\.cl\/hogar\/internet\-fibra\-optica\/conoce\-movistar\-fibra\/.*|.*ww2.movistar.cl\/mifibra-v2.*|.*ww2.movistar.cl(\/inicio)?\/vURL\/Ventas\/(Clicktocall|formulario)\/19060\/.*|ofertas.movistar.cl.*hogar.*tab=(home|banda-ancha|duo|trio)|ofertas.movistar.cl\/hogar\/[^/?]+(\/)?$).*"
  filtro_movil = "(?i).*(ww2\.movistar\.cl\/movil\/(planes-familia|planes-portabilidad|movistarone|planes-multilineas|planes-2x1)|catalogo\.movistar\.cl\/(equipomasplan)|catalogo\.movistar\.cl\/movil\/solicitud-web|ww2\.movistar\.cl\/(ofertas|ofertas.*)\/(planes-portabilidad|equipo-plan|equipos)|ww2\.movistar\.cl\/(movistarone\/equipos|equipos\/movistarone)|ww2\.movistar\.cl\/hogar\/wifi-movil(\?|$)|ww2.movistar.cl/equipos($|/(apple|xiaomi|samsung|huawei|moto|lg|vivo|tcl))|ww2.movistar.cl/cyber/portabilidad/(equipo-plan|mone|planes-2x1)|ww2.movistar.cl(/inicio)?/Ventas/(Clicktocall|formulario)/19059/|catalogo.movistar.cl/planes/solicitud/detalle|catalogo.movistar.cl/(tienda|celulares|detalle-producto).*/vURL/equipomasplan-(portabilidad|nuevo_numero)|ofertas.movistar.cl(\/movil|.*tab=(movil|con-equipo))|ww2.movistar.cl/soycliente/movil).*"
  
  filtro_fullprice = ".*catalogo.movistar.cl/(celulares|fullprice|detalle-producto|tienda).*|.*ww2.movistar.cl/(equipos|ofertas)/(celulares-liberados|seminuevos|accesorios).*|[^?]*(ipad|accesorio|smartwatch|notebook|gaming|airtag|router|charger|smart_tv|ps5|playstation|scooter).*"

  #
  filtro_recambio = "(?i).*ww2.movistar.cl\/ofertas\/renovacion-movil.*|.*catalogo.movistar.cl\/equipomasplan\/.*\/vURL\/renovacion.*|.*catalogo.movistar.cl\/(tienda|celulares|detalle-producto).*\/vURL\/equipomasplan-renovacion.*|.*catalogo.movistar.cl\/equipomasplan\/recambio.*|.*cyber\/renovacion\/equipos-renovacion.*|.*ww2.movistar.cl\/movil\/planes-renovacion(-v[0-9])?\/Ventas\/Clicktocall.*|.*ofertas.movistar.cl\/renovacion.*|.*ww2.movistar.cl\/ofertas\/renovacion-app.*|.*movistar.cl\/.*tienda-app\/.*tienda-equipos\/modo-pago-(boleta|movistarone|tarjeta).*|.*ofertas.movistar.cl\/formularios\/mi-movistar\/recambio.*"

  filtro_list_trafico = [filtro_hogar, filtro_movil, filtro_fullprice, filtro_recambio]

  ## REGEXs para extraer métrica interes ##
  filtro_interes_hogar = "(?i).*catalogo.movistar.cl\/servicioshogar\/contratacion\/datos\/index\/id\/.*"
  filtro_interes_movil = "(?i).*catalogo.movistar.cl\/planes\/(solicitud|contratacion|upselling)\/detalle\/index\/id/.*"
  filtro_interes_fullprice = "(?i).*catalogo.movistar.cl\/(fullprice|tienda)\/checkout(\/)?(\?.*)?$"
  filtro_interes_recambio = "No Aplica"

  filtro_list_interes = [filtro_interes_hogar, filtro_interes_movil, filtro_interes_fullprice, filtro_interes_recambio]

  # Desbloquear las 2 siguientes lineas en caso de cargar una(s) fecha(s) en específico
  fecha_inicio = datetime.strptime(fecha_tmp_ejecucion_str, "%Y-%m-%d").date()  # Escribir directamente fecha en formato Año-Mes-Día
  fecha_termino = datetime.strptime(fecha_tmp_ejecucion_str, "%Y-%m-%d").date() # Escribir directamente fecha en formato Año-Mes-Día (Estaba originalmente el 2025-01-03)

  # 5. Query GA4
  segment_list = ["Hogar", "Movil", "Fullprice", "Recambio"] # opcional para print de avance

  df_base = pd.DataFrame(columns= dims + metrics)
  df_list = [df_base, df_base, df_base, df_base]

  start_time = time.time()

  #print("Inicio Query Tráfico")
  for i in range(0, (fecha_termino - fecha_inicio).days + 1):
    fecha = (fecha_inicio + timedelta(days=i)).strftime("%Y-%m-%d")
    for i, df in enumerate(df_list):
      data_trafico = query_ga(property_id, fecha, fecha, dims, metrics, filtro_list_trafico[i])
      data_session_conversion = query_ga_conversions(property_id, fecha, fecha, dims, [metrics_sessions_conversion[i]])
      data_session_interes = query_ga(property_id, fecha, fecha, dims, metrics_sessions_interes, filtro_list_interes[i])

      df_trafico = crear_dataframe(data_trafico)
      df_session_conversion = crear_dataframe(data_session_conversion)
      df_session_interes = crear_dataframe(data_session_interes)

      df_session_interes = df_session_interes.rename(columns={'sessions': 'SessionInteres'})

      merged_df = pd.merge(df_trafico, df_session_interes, on=dims, how='outer')
      merged_df = pd.merge(merged_df, df_session_conversion, on=dims, how='outer')
      merged_df = merged_df.fillna(0)

      df = pd.concat([df, merged_df], ignore_index=True)
      df['segment'] = segment_list[i]
      df_list[i] = df
    # Guardar archivos crudos
    #print("Datos del día {} extraídos".format(fecha))


  ## 6. Transformación y Carga de Data
  df_acumulado = pd.DataFrame()

  for i, df in enumerate(df_list): # Loop para todos los negocios
    # Cambiar formato de las métricas y la fecha
    df[metrics + [metrics_sessions_conversion[i]] + ['SessionInteres']] = df[metrics + [metrics_sessions_conversion[i]] + ['SessionInteres']].astype(int)

    df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
    df = df.sort_values(by=['date', 'sessions'], ascending=[True, False])
    df_list[i] = df

    # Renombrar métrica conversiones
    new_columns = df.columns.tolist()
    new_columns[-1] = 'SessionKeyEvent' # SessionKeyEvent
    df = df.rename(columns=dict(zip(df.columns, new_columns)))

    # Renombrar dimensión de canales (porque tiene carácter extraño)
    df = df.rename(columns={'sessionCustomChannelGroup:5602220125': 'canalesVentas'})

    # Arreglos para que el export sea igual al presentado por equipo: (habían diferencias de nombres en columnas)
    df = df.rename(columns={'SessionKeyEvent': 'negocio'})
    df = df.rename(columns={'SessionInteres': 'interesAuto'})

    if 'keyEvents:Obj6_Leads_Hogar' in df.columns:
      df = df.rename(columns={'keyEvents:Obj6_Leads_Hogar': 'SessionKeyEvent'})

    if 'keyEvents:Obj7_Leads_Movil' in df.columns:
      df = df.rename(columns={'keyEvents:Obj7_Leads_Movil': 'SessionKeyEvent'})

    if 'keyEvents:Obj3_Comprobante_Fullprice' in df.columns:
      df = df.rename(columns={'keyEvents:Obj3_Comprobante_Fullprice': 'SessionKeyEvent'})

    if 'keyEvents:Obj19_Leads_Recambio' in df.columns:
      df = df.rename(columns={'keyEvents:Obj19_Leads_Recambio': 'SessionKeyEvent'})

    # CONCAT
    df_acumulado = pd.concat([df_acumulado, df], ignore_index = True)

    # PYSPARK 
  df_acumulado_ = df_acumulado[["date", "sessionSource", "sessionMedium", "sessionCampaignName", "canalesVentas", "deviceCategory", "negocio", "sessions", "engagedSessions", "totalUsers", "activeUsers", "interesAuto", "SessionKeyEvent"]].copy()
  # Agregar columnas year / month

  df_spark = spark.createDataFrame(df_acumulado_)

  # FECHAS DE CONTROL 

  df_spark = df_spark.withColumn("year", year(to_date(lit(fecha_tmp_ejecucion_str), "yyy-MM-dd"))).withColumn("month", month(to_date(lit(fecha_tmp_ejecucion_str), "yyy-MM-dd")))

  export = df_spark.withColumn("year",col("year").cast("string")).withColumn("month", format_string("%02d", col("month")).cast("string")).withColumn("bigdata_close_date", current_timestamp().cast("timestamp"))

  ##Creando el valor de Bigdata ctrl_id con la fecha actual
  try:
      current_time = datetime.now()  # O usando time.time() para segundos desde la época
      print(current_time)
      formatted_time = current_time.strftime("%Y%m%d%H%M%S")
      counter = "001"
      bigdata_ctrl_id = "{}{}".format(formatted_time,counter)
      print(bigdata_ctrl_id)
  except Exception as e:
      # Capturar cualquier excepción y mostrar un mensaje de error
      status = "[ERROR]"
      desc_status = f"[ERROR] : {e}"
      print(status)
      print(desc_status)    
      print(f"Se produjo un error: {e}")

  try:
      export = export \
          .withColumn("bigdata_close_date", current_timestamp()) \
          .withColumn("bigdata_ctrl_id", lit(bigdata_ctrl_id))
  
      export = export \
          .withColumn("bigdata_ctrl_id", col("bigdata_ctrl_id").cast("bigint"))
  
  except Exception as e:
      # Capturar cualquier excepción y mostrar un mensaje de error
      status = "[ERROR]"
      desc_status = f"[ERROR] : {e}"

  # EXPORT
  orden_columnas = ["date", "sessionSource", "sessionMedium", "sessionCampaignName", "canalesVentas", "deviceCategory", "negocio", "sessions", "engagedSessions", "totalUsers", "activeUsers", "interesAuto", "SessionKeyEvent", "bigdata_close_date", "bigdata_ctrl_id", "year", "month"]

  schema_export = StructType([StructField("date", StringType(), True), StructField("sessionSource", StringType(), True), StructField("sessionMedium", StringType(), True), StructField("sessionCampaignName", StringType(), True), StructField("canalesVentas", StringType(), True), StructField("deviceCategory", StringType(), True), StructField("negocio", StringType(), True), StructField("sessions", LongType(), True), StructField("engagedSessions", LongType(), True), StructField("totalUsers", LongType(), True), StructField("activeUsers", LongType(), True), StructField("interesAuto", LongType(), True), StructField("SessionKeyEvent", LongType(), True), StructField("bigdata_close_date", TimestampType(), True), StructField("bigdata_ctrl_id", LongType(), True), StructField("year", StringType(), True), StructField("month", StringType(), True)])
                             
  export_ordenado = export.select(orden_columnas)
  export_ordenado = export_ordenado.select(
    col("date").cast(StringType()).alias("date"), 
    col("sessionSource").cast(StringType()).alias("sessionSource"),
    col("sessionMedium").cast(StringType()).alias("sessionMedium"),
    col("sessionCampaignName").cast(StringType()).alias("sessionCampaignName"),
    col("canalesVentas").cast(StringType()).alias("canalesVentas"),
    col("deviceCategory").cast(StringType()).alias("deviceCategory"),
    col("negocio").cast(StringType()).alias("negocio"),
    col("sessions").cast(LongType()).alias("sessions"),
    col("engagedSessions").cast(LongType()).alias("engagedSessions"),
    col("totalUsers").cast(LongType()).alias("totalUsers"),
    col("activeUsers").cast(LongType()).alias("activeUsers"),
    col("interesAuto").cast(LongType()).alias("interesAuto"),
    col("SessionKeyEvent").cast(LongType()).alias("SessionKeyEvent"),
    col("bigdata_close_date").cast(TimestampType()).alias("bigdata_close_date"),
    col("bigdata_ctrl_id").cast(LongType()).alias("bigdata_ctrl_id"),
    col("year").cast(StringType()).alias("year"),
    col("month").cast(StringType()).alias("month")
  )
  
  try:
    # export de solamente reemplazar el día usado en la particion mensual
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

    anio = fecha_tmp_ejecucion.year
    mes = fecha_tmp_ejecucion.month
    dia = fecha_tmp_ejecucion.strftime("%Y-%m-%d")
    tabla_delta = DeltaTable.forPath(spark, f"{dir_adls}raw/")
    tabla_delta.delete(
      (col("year") == anio) & 
      (col("month") == mes) &
      (col("date") == dia)
    )
    
    #print ("extraccion completa")
    export_ordenado.write.format("delta").mode("append").partitionBy("year", "month").option("path", f"{dir_adls}raw/").option("mergeSchema", "true").saveAsTable(tabla_vw)
    
    if k == 0:
      final_row_count = export_ordenado.count()

  except Exception as e:
    status = "[ERROR]"
    desc_status = f"[ERROR] : {e}"

In [0]:
original_row_count = 0
data_source_name = None
original_file_date = None
data_source_type = "table"
endtime_spark = endtime_nifi    
starttime_spark = starttime_nifi
process_type = "normal"
final_file_size = 0
big_data_ctrl_id = str(bigdata_ctrl_id) 
#print ("2 starttime_nifi: ", starttime_nifi)

dif_row_count = int(final_row_count) - int(original_row_count)

original_file_size = int(original_file_size)
final_file_size = str(final_file_size)
original_row_count = None
final_row_count = str(final_row_count)
dif_row_count = str(dif_row_count)

final_number_of_files = str(original_row_count)
end_file_name = table_name
insert_data_ctrl_date = datetime.now().strftime("%Y-%m-%d")
hdfs_path = f"{dir_adls}raw/"

#Formato de Variable con Fechas
#original_file_datef = datetime.strptime(original_file_date, "%Y-%m-%dT%H:%M:%SZ")
#original_file_date2 = original_file_datef.strftime("%Y-%m-%d %H:%M:%S")

#original_file_datef = datetime.strptime(original_file_date, "%Y-%m-%dT%H:%M:%SZ")
#original_file_date2 = original_file_datef.strftime("%Y-%m-%d %H:%M:%S")

# Utilizamos substring para extraer las partes de la fecha y hora
anio = starttime_nifi_datetime.strftime("%Y")
mes = starttime_nifi_datetime.strftime("%m")
dia = starttime_nifi_datetime.strftime("%d")
hora = starttime_nifi_datetime.strftime("%H")
minuto = starttime_nifi_datetime.strftime("%M")
segundo = starttime_nifi_datetime.strftime("%S")
## Formateamos la fecha en el nuevo formato
starttime_nifi2 = f"{anio}-{mes}-{dia} {hora}:{minuto}:{segundo}"

# Utilizamos substring para extraer las partes de la fecha y hora
anio = endtime_nifi_datetime.strftime("%Y")
mes = endtime_nifi_datetime.strftime("%m")
dia = endtime_nifi_datetime.strftime("%d")
hora = endtime_nifi_datetime.strftime("%H")
minuto = endtime_nifi_datetime.strftime("%M")
segundo = endtime_nifi_datetime.strftime("%S")
endtime_nifi2 = f"{anio}-{mes}-{dia} {hora}:{minuto}:{segundo}"

# Convert strings to datetime objects
starttime_nifi_dt = datetime.strptime(starttime_nifi2, "%Y-%m-%d %H:%M:%S")
endtime_nifi_dt = datetime.strptime(endtime_nifi2, "%Y-%m-%d %H:%M:%S")

# Calculate the difference
totaltime_nifi_diff = endtime_nifi_dt - starttime_nifi_dt
totaltime_nifi_diff_seconds = totaltime_nifi_diff.total_seconds()

totaltime_nifi = str(totaltime_nifi_diff_seconds)

totaltime_spark = float(totaltime_nifi_diff_seconds)
totaltime_process = float(totaltime_nifi_diff_seconds)

data_control = [(big_data_ctrl_id, process_name, data_source_type, data_source_name, original_file_date,
starttime_nifi2, endtime_nifi2, totaltime_nifi, starttime_spark, endtime_spark ,
totaltime_spark, totaltime_process, insert_data_ctrl_date, process_type, 
original_file_size, final_file_size, original_row_count, final_row_count, 
dif_row_count, final_number_of_files, end_file_name, hdfs_path,
pipelineRunId, status, desc_status,  bigdata_ctrl_id )]

schema_control = StructType([StructField("big_data_ctrl_id", StringType(), True), StructField("process_name", StringType(), True), StructField("data_source_type", StringType(), True), StructField("data_source_name", StringType(), True), StructField("original_file_date", StringType(), True), StructField("starttime_nifi", StringType(), True), StructField("endtime_nifi", StringType(), True), StructField("totaltime_nifi", StringType(), True), StructField("starttime_spark", StringType(), True), StructField("endtime_spark", StringType(), True), StructField("totaltime_spark", FloatType(), True), StructField("totaltime_process", FloatType(), True), StructField("insert_data_ctrl_date", StringType(), True), StructField("process_type", StringType(), True), StructField("original_file_size", StringType(), True), StructField("final_file_size", StringType(), True), StructField("original_row_count", StringType(), True), StructField("final_row_count", StringType(), True), StructField("dif_row_count", StringType(), True), StructField("final_number_of_files", StringType(), True), StructField("end_file_name", StringType(), True), StructField("hdfs_path", StringType(), True), StructField("pipelineRunId", StringType(), True), StructField("status", StringType(), True), StructField("desc_status", StringType(), True), StructField("bigdata_ctrl_id", StringType(), True)])

df_data_control = spark.createDataFrame(data_control, schema = schema_control)

df_data_control = df_data_control \
.withColumn("starttime_nifi", from_unixtime(unix_timestamp(col("starttime_nifi"), "yyyy-MM-dd HH:mm:ss"), "yyyy-MM-dd HH:mm:ss")) \
.withColumn("endtime_nifi", from_unixtime(unix_timestamp(col("endtime_nifi"), "yyyy-MM-dd HH:mm:ss"), "yyyy-MM-dd HH:mm:ss")) \
.withColumn("totaltime_spark", col("totaltime_spark").cast("float")) \
.withColumn("totaltime_process", col("totaltime_process").cast("float")) \
.withColumn("big_data_ctrl_id", col("big_data_ctrl_id").cast("string")) \
.withColumn("bigdata_ctrl_id", col("bigdata_ctrl_id").cast("string"))
#.withColumn("original_file_date", from_unixtime(unix_timestamp(col("original_file_date"), "yyyy-MM-dd HH:mm:ss"), "yyyy-MM-dd HH:mm:ss")) \

df_data_control.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(catalog_control)